<a href="https://colab.research.google.com/github/Prasanna-Mahajan-2006/Deepfake-Detector/blob/main/FFT_%2B_ATTENTION_(EXPERIMENTAL_NOTEBOOK_4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub

In [ ]:
#importing necessary libraries

import os
import cv2
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
!pip install kagglehub -q

import kagglehub

# downloading the dataset


print("Downloading dataset via kagglehub...")
# This will download the dataset and save the exact folder location to the 'path' variable
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")

print(f"Success! Dataset downloaded to: {path}")

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Peek inside the downloaded directory to confirm the folder name (usually 'Dataset')
print("Contents of the downloaded path:", os.listdir(path))


In [ ]:
# Extracting the datasets into variables
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x)

# --- NEW: Squeeze-and-Excitation Attention Block ---
# 1. Squeeze step: Compress spatial dimensions
attention_weights = layers.GlobalAveragePooling2D()(x)
# 2. Excitation step: Learn the channel weights (using a reduction ratio of 16)
channels = x.shape[-1]
attention_weights = layers.Dense(channels // 16, activation='relu')(attention_weights)
attention_weights = layers.Dense(channels, activation='sigmoid')(attention_weights)
# 3. Reshape and apply weights to the original feature map
attention_weights = layers.Reshape((1, 1, channels))(attention_weights)
x = layers.Multiply()([x, attention_weights])
# ---------------------------------------------------

# Continue with your existing architecture
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(0.01))(x)

model = keras.Model(inputs, outputs)

In [ ]:
# Implementing the attention + FFT implementation

# ==========================================
# 1. ATTENTION MECHANISM (CBAM/SE Hybrid)
# ==========================================
# The paper emphasizes using attention to focus on relevant manipulation regions.
def channel_spatial_attention(input_feature, ratio=8):
    """Applies a lightweight Attention mechanism to the feature map."""
    channel_axis = 1 if tf.keras.backend.image_data_format() == "channels_first" else -1
    filters = input_feature.shape[channel_axis]

    # Channel Attention
    shared_layer_one = layers.Dense(filters // ratio, activation='relu', kernel_initializer='he_normal', use_bias=True)
    shared_layer_two = layers.Dense(filters, kernel_initializer='he_normal', use_bias=True)

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, filters))(avg_pool)
    avg_pool = shared_layer_two(shared_layer_one(avg_pool))

    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, filters))(max_pool)
    max_pool = shared_layer_two(shared_layer_one(max_pool))

    cbam_feature = layers.Add()([avg_pool, max_pool])
    cbam_feature = layers.Activation('sigmoid')(cbam_feature)

    return layers.Multiply()([input_feature, cbam_feature])

# ==========================================
# 2. HYBRID MODEL ARCHITECTURE
# ==========================================
def build_fft_attention_mobilenet(img_size=(224, 224)):
    """Builds the dual-stream model integrating MobileNet, Attention, and FFT[cite: 1]."""

    # --- STREAM 1: SPATIAL DOMAIN (MobileNet + Attention) ---
    # Standard RGB input for MobileNet
    spatial_input = layers.Input(shape=img_size + (3,), name="spatial_rgb_input")

    # Lightweight MobileNet backbone for spatial feature extraction[cite: 1]
    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=img_size + (3,),
        include_top=False,
        weights='imagenet'
    )
    # Unfreeze top layers for fine-tuning
    base_model.trainable = True

    x_spatial = base_model(spatial_input)

    # Apply Attention Layer to enhance artifact sensitivity[cite: 1]
    x_spatial = channel_spatial_attention(x_spatial)
    x_spatial = layers.GlobalAveragePooling2D()(x_spatial)
    x_spatial = layers.BatchNormalization()(x_spatial)

    # --- STREAM 2: FREQUENCY DOMAIN (FFT) ---
    # Grayscale FFT Magnitude input. Preprocessing converts images to grayscale and applies FFT[cite: 1].
    freq_input = layers.Input(shape=img_size + (1,), name="frequency_fft_input")

    # Lightweight CNN to process the spectral patterns[cite: 1]
    x_freq = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(freq_input)
    x_freq = layers.MaxPooling2D((2, 2))(x_freq)
    x_freq = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x_freq)
    x_freq = layers.MaxPooling2D((2, 2))(x_freq)
    x_freq = layers.GlobalAveragePooling2D()(x_freq)
    x_freq = layers.BatchNormalization()(x_freq)

    # --- MERGING STREAMS ---
    # Combine spatial and frequency features to capture anomalies invisible in the spatial domain[cite: 1]
    combined = layers.Concatenate()([x_spatial, x_freq])

    # Dropout for generalization
    combined = layers.Dropout(0.4)(combined)

    # Final classification layer (Binary: Real vs Fake)
    outputs = layers.Dense(1, activation='sigmoid')(combined)

    model = models.Model(inputs=[spatial_input, freq_input], outputs=outputs, name="FFT_Attention_MobileNet")

    # The study utilizes the Adam Optimizer for training[cite: 1]
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
    )

    return model

# Instantiate the model
hybrid_model = build_fft_attention_mobilenet()
hybrid_model.summary()

In [ ]:
def generate_hybrid_inputs(image_path, img_size=(224, 224)):
    """
    Reads an image and generates both the RGB array and the FFT magnitude spectrum.
    """
    # 1. Load standard RGB image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_rgb = cv2.resize(img_rgb, img_size)
    img_rgb = img_rgb / 255.0 # Normalize spatial data

    # 2. Process Frequency Domain (FFT)[cite: 1]
    # Convert to grayscale to reduce dimensionality[cite: 1]
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_gray = cv2.resize(img_gray, img_size)

    # Apply Fast Fourier Transform[cite: 1]
    f_transform = np.fft.fft2(img_gray)
    # Shift the zero-frequency component to the center of the spectrum
    f_shift = np.fft.fftshift(f_transform)

    # Calculate magnitude spectrum and use log scale to make patterns visible
    # Adding a small epsilon (1e-8) prevents log(0) errors
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1e-8)

    # Normalize the frequency data[cite: 1]
    magnitude_spectrum = (magnitude_spectrum - np.min(magnitude_spectrum)) / (np.max(magnitude_spectrum) - np.min(magnitude_spectrum))

    # Reshape for Keras (Height, Width, Channels)
    magnitude_spectrum = np.expand_dims(magnitude_spectrum, axis=-1)

    return img_rgb, magnitude_spectrum

In [ ]:
# 1. SETUP VARIABLES
DATASET_PATH = f"{path}/Dataset"
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 15

# ==========================================
# 2. CUSTOM DUAL-STREAM DATA GENERATOR (UPDATED)
# ==========================================
class DualStreamGenerator(Sequence):
    def __init__(self, directory, batch_size=32, img_size=(224, 224), shuffle=True):
        self.directory = directory
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.filepaths = []
        self.labels = []

        # Scan directory and map Real (0) and Fake (1)
        for label, class_name in enumerate(['Real', 'Fake']):
            class_dir = os.path.join(directory, class_name)
            if os.path.exists(class_dir):
                for fname in os.listdir(class_dir):
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.filepaths.append(os.path.join(class_dir, fname))
                        self.labels.append(label)

        self.filepaths = np.array(self.filepaths)
        self.labels = np.array(self.labels)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.filepaths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_paths = self.filepaths[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_labels = self.labels[idx * self.batch_size:(idx + 1) * self.batch_size]

        X_spatial, X_freq, Y_valid = [], [], []

        for i, file_path in enumerate(batch_paths):
            img = cv2.imread(file_path)
            if img is None:
                continue

            # --- 1. SPATIAL STREAM (RGB) ---
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_rgb = cv2.resize(img_rgb, self.img_size)
            img_rgb = img_rgb / 255.0

            # --- 2. FREQUENCY STREAM (FFT) ---
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img_gray = cv2.resize(img_gray, self.img_size)
            f_transform = np.fft.fft2(img_gray)
            f_shift = np.fft.fftshift(f_transform)

            magnitude = 20 * np.log(np.abs(f_shift) + 1e-8)
            magnitude = (magnitude - np.min(magnitude)) / (np.max(magnitude) - np.min(magnitude) + 1e-8)
            magnitude = np.expand_dims(magnitude, axis=-1)

            X_spatial.append(img_rgb)
            X_freq.append(magnitude)
            Y_valid.append(batch_labels[i])

        # ========================================================
        # THE FIX: Return a Dictionary mapping names to arrays
        # ========================================================
        return {
            "spatial_rgb_input": np.array(X_spatial),
            "frequency_fft_input": np.array(X_freq)
        }, np.array(Y_valid)

    def on_epoch_end(self):
        if self.shuffle:
            indices = np.arange(len(self.filepaths))
            np.random.shuffle(indices)
            self.filepaths = self.filepaths[indices]
            self.labels = self.labels[indices]

# ==========================================
# 3. INITIALIZE & TRAIN
# ==========================================
print("Preparing Dual-Stream Data Generators...")
train_gen = DualStreamGenerator(f"{DATASET_PATH}/Train", batch_size=BATCH_SIZE)
val_gen = DualStreamGenerator(f"{DATASET_PATH}/Validation", batch_size=BATCH_SIZE, shuffle=False)

print("Starting training! (This may take longer per epoch due to live FFT calculations)")

callbacks = [
    keras.callbacks.ModelCheckpoint("best_hybrid_model.keras", save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_loss')
]

history = hybrid_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("\nTraining Complete! Model saved as 'best_hybrid_model.keras'")